In [ ]:
# ===== Imports and Initialization =====
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from tavily import TavilyClient
from sklearn.metrics.pairwise import cosine_similarity
import getpass
import os
from dotenv import load_dotenv
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline
from huggingface_hub import InferenceClient
from time import sleep
import random
from sentence_transformers import SentenceTransformer, util

load_dotenv()

# Initialize clients and models
hf_client = InferenceClient(token=os.getenv("HUGGINGFACE_API_TOKEN"))
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ===== Shared Data Models =====
class CorpusMetrics(BaseModel):
    """Metrics for corpus quality assessment"""
    coverage: float = Field(description="Coverage score of corpus relative to question")
    relevance: float = Field(description="Relevance score of corpus")
    diversity: float = Field(description="Diversity score of corpus")
    size: int = Field(description="Number of documents in corpus")

# ===== Shared Utility Functions =====
def get_similarity_score(text1: str, text2: str) -> float:
    """Get semantic similarity score using sentence transformers"""
    try:
        emb1 = embedder.encode(text1, convert_to_tensor=True)
        emb2 = embedder.encode(text2, convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(emb1, emb2)
        return float(similarity[0][0])
    except Exception as e:
        print(f"Warning: Error getting similarity score: {e}")
        return 0.0

def summarize_docs(docs: List[str]) -> str:
    """Create a brief summary of the documents"""
    if not docs:
        return "No documents available"
    return "\n".join(f"- {doc.page_content[:200]}..." for doc in docs)

# ===== MULTI-HOP RETRIEVAL SECTION =====
def decompose_question(llm: ChatOpenAI, question: str) -> List[str]:
    """Break down complex question into one focused subquery"""
    subquery_prompt = f"""
    Decompose this question into ONE more focused search query that could help answer it.
    Focus on the most important aspect needed for a complete answer.
    Question: {question}
    Subquestion:
    """
    subquery_output = llm.invoke(subquery_prompt).content.strip()
    return [subquery_output]

def evaluate_corpus_quality(question: str, documents: List[str]) -> CorpusMetrics:
    """Evaluate the quality of the current corpus using similarity scoring"""
    if not documents:
        return CorpusMetrics(coverage=0.0, relevance=0.0, diversity=0.0, size=0)
    
    similarity_scores = [
        get_similarity_score(doc.page_content[:512], question)
        for doc in documents
    ]
    
    coverage_score = sum(similarity_scores) / len(similarity_scores)
    
    embeddings = OpenAIEmbeddings()
    if len(documents) > 1:
        doc_embeddings = embeddings.embed_documents([doc.page_content for doc in documents])
        similarities = cosine_similarity([doc_embeddings[0]], doc_embeddings[1:])
        diversity_score = 1.0 - (similarities.mean())
    else:
        diversity_score = 1.0
    
    return CorpusMetrics(
        coverage=coverage_score,
        relevance=coverage_score,
        diversity=diversity_score,
        size=len(documents)
    )

def search_and_retrieve(tavily: TavilyClient, question: str, llm: ChatOpenAI) -> List[str]:
    """Search and retrieve URLs for a given question"""
    subqueries = decompose_question(llm, question)
    urls = []
    
    for q in subqueries:
        print(f"\nSearching: {q}")
        results = tavily.search(q, max_results=3)["results"]
        urls.extend([r["url"] for r in results])
    
    return urls

def build_multi_hop_corpus(
    question: str,
    llm: ChatOpenAI,
    tavily: TavilyClient,
    max_hops: int = 3,
    quality_threshold: float = 0.5,
    max_questions_per_hop: int = 3
) -> tuple[List[str], List[Dict]]:
    all_urls = []
    hop_history = []
    best_score = 0.0
    
    for hop in range(max_hops):
        print(f"\n=== Processing Hop {hop + 1} ===")
        hop_urls = []
        
        if hop == 0:
            current_questions = [question]
        else:
            if best_score >= quality_threshold:
                break
                
            gap_prompt = f"""
            Based on current information (score: {best_score}),
            what are the {max_questions_per_hop} MOST IMPORTANT questions
            needed to understand: "{question}"?
            """
            current_questions = llm.invoke(gap_prompt).content.strip().split('\n')[:max_questions_per_hop]
        
        for sub_q in current_questions:
            print(f"\nSearching: {sub_q}")
            new_urls = search_and_retrieve(tavily, sub_q, llm)
            
            # Load documents temporarily for evaluation
            temp_docs = [WebBaseLoader(url).load() for url in new_urls]
            temp_docs = [item for sublist in temp_docs for item in sublist]
            metrics = evaluate_corpus_quality(sub_q, temp_docs)
            
            if metrics.coverage > best_score:
                best_score = metrics.coverage
                hop_urls.extend(new_urls)
                print(f"New best coverage score: {best_score:.2f}")
                
            if best_score >= quality_threshold:
                print(f"Reached sufficient coverage ({best_score:.2f})")
                break
        
        hop_summary = {
            "hop": hop,
            "questions": current_questions,
            "urls_found": len(hop_urls),
            "metrics": metrics
        }
        hop_history.append(hop_summary)
        
        all_urls.extend(hop_urls)
        
        # Load documents temporarily for completeness check
        temp_docs = [WebBaseLoader(url).load() for url in all_urls]
        temp_docs = [item for sublist in temp_docs for item in sublist]
        
        completeness_check = llm.invoke(f"""
        Based on all information gathered:
        {summarize_docs(temp_docs)}
        
        Can we fully answer: "{question}"?
        Answer with either 'COMPLETE' or 'INCOMPLETE':
        """).content.strip()
        
        if completeness_check == "COMPLETE":
            print(f"Question can be fully answered after {hop + 1} hops")
            break
    
    return all_urls, hop_history



